In [ ]:
# this notebook conduct the following task:
# 1 get antigen surface index tsv files grom antigen_pdb_with_index
#   pdb_input_dir = "/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/antigen_pdb_with_SI"
#   tsv_output_dir = "/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/antigen_info_SI"

# 2 concatenate the SI information from antigen_info_SI/ to antigen_info/
#   si_input_dir = "/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/antigen_info_SI"
#   interaction_input_dir = "/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/antigen_info"
#   output_dir = "/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/antigen_info_SI_interaction"

In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
## for the first-time execution of this code
## install the packages in google Drive locally (/content/drive/MyDrive/Colab Notebooks/antigen_antibody/packages/)
import os
import sys

nb_path = '/content/notebooks'

if not os.path.exists(nb_path):
    os.symlink('/content/drive/MyDrive/@antibody/antigen_epitope/abdb_v2/packages', nb_path)

sys.path.insert(0, nb_path)

In [ ]:
!pip install biopython --target=$nb_path

In [ ]:
## non-first-time execution
## package was previously installed through process_data_positive_patch_3points-Graph.ipynb
import sys
sys.path.append('/content/drive/MyDrive/@antibody/antigen_epitope/abdb_v2/packages')

In [ ]:
import os
import numpy as np
import pandas as pd
from Bio import PDB

In [ ]:
#@title Get antigen surface index tsv files from antigen_pdb_with_index

from Bio import PDB
import pandas as pd
import os

# Define the input and output directories
pdb_input_dir = "/content/drive/My Drive/@antigen_epitope/abdb_v3/data_test/antigen_pdb_with_SI"
tsv_output_dir = "/content/drive/My Drive/@antigen_epitope/abdb_v3/data_test/antigen_info_SI"

!rm -r "/content/drive/My Drive/@antigen_epitope/abdb_v3/data_test/antigen_info_SI"
os.makedirs(tsv_output_dir, exist_ok=True)

# Initialize PDB Parser
parser = PDB.PDBParser()

# Function to calculate average B-factor for a residue
def average_b_factor(residue):
    b_factors = [atom.get_bfactor() for atom in residue]
    return sum(b_factors) / len(b_factors) if b_factors else None

# Process each PDB file in the input directory
for pdb_filename in os.listdir(pdb_input_dir):
    if pdb_filename.endswith('.pdb'):
        pdb_filepath = os.path.join(pdb_input_dir, pdb_filename)
        # Extract the PDB ID and the chain part from the filename, excluding '_antigen_updated'
        structure_id_parts = pdb_filename.split('_')[:-2]
        structure_id = "_".join(structure_id_parts)
        structure = parser.get_structure(structure_id, pdb_filepath)

        # List to hold the data
        data = []

        # Iterate over all residues in the structure
        for model in structure:
            for chain in model:
                for residue in chain:
                    if PDB.is_aa(residue, standard=True):  # Check if it's a standard amino acid
                        # Get the residue ID as a tuple (hetatm flag, residue sequence number, insertion code)
                        res_id_tuple = residue.get_id()
                        # Combine tuple elements to create a unique residue ID string
                        res_id = f"{res_id_tuple[0]}{res_id_tuple[1]}{res_id_tuple[2]}".strip()
                        res_name = residue.resname
                        si = average_b_factor(residue)
                        # Append the residue info to the data list
                        data.append([structure_id, res_id, res_name, si])

        # Convert the data into a pandas DataFrame
        df = pd.DataFrame(data, columns=['name', 'ResID', 'AA', 'SI'])

        # Save the DataFrame to a TSV file
        tsv_filename = f"{structure_id}_info.tsv"
        tsv_filepath = os.path.join(tsv_output_dir, tsv_filename)
        df.to_csv(tsv_filepath, sep='\t', index=False)
        print(f"Processed {pdb_filename} and saved to {tsv_filename}")


In [ ]:
!ls "/content/drive/My Drive/@antigen_epitope/abdb_v3/data_test/antigen_info_SI"

In [ ]:
#@title Concatenate the SI information from antigen_info_SI/ to antigen_info/

import os
import pandas as pd

# Define the input directories
si_input_dir = "/content/drive/My Drive/@antigen_epitope/abdb_v3/data_test/antigen_info_SI"
interaction_input_dir = "/content/drive/My Drive/@antigen_epitope/abdb_v3/data_test/antigen_info"

# Define the output directory
output_dir = "/content/drive/My Drive/@antigen_epitope/abdb_v3/data_test/antigen_info_SI_interaction"
os.makedirs(output_dir, exist_ok=True)

# Process each TSV file in the SI input directory
for tsv_filename in os.listdir(si_input_dir):
    si_filepath = os.path.join(si_input_dir, tsv_filename)
    interaction_filepath = os.path.join(interaction_input_dir, tsv_filename)

    # Check if the corresponding interaction file exists
    if os.path.exists(interaction_filepath):
        # Read both files into dataframes
        si_df = pd.read_csv(si_filepath, sep='\t')
        interaction_df = pd.read_csv(interaction_filepath, sep='\t')

        # Ensure 'ResID' and 'AA' are of type string for both dataframes
        si_df['ResID'] = si_df['ResID'].astype(str)
        interaction_df['ResID'] = interaction_df['ResID'].astype(str)
        si_df['AA'] = si_df['AA'].astype(str)
        interaction_df['AA'] = interaction_df['AA'].astype(str)

        # Merge the dataframes on 'ResID' and 'AA'
        merged_df = pd.merge(interaction_df, si_df[['ResID', 'AA', 'SI']], on=['ResID', 'AA'], how='left')

        # Save the merged dataframe to a new TSV file in the output directory
        output_filepath = os.path.join(output_dir, tsv_filename)
        merged_df.to_csv(output_filepath, sep='\t', index=False)
        print(f"Merged file saved to {output_filepath}")
    else:
        print(f"No corresponding interaction file found for {tsv_filename}.")


In [ ]:
import os

# Define the source and target directories
target_directory = '/content/drive/My Drive/@antigen_epitope/abdb_v3/data_test/antigen_info_SI_interaction'

def count_files_in_folder(folder_path):
    items = os.listdir(folder_path)
    file_count = sum(1 for item in items if os.path.isfile(os.path.join(folder_path, item)))
    return file_count

print(count_files_in_folder(target_directory))

In [ ]:
#@title Merge all the tsv files together

In [ ]:
import os
import pandas as pd

# Define the output directory where TSV files are located
output_dir = "/content/drive/My Drive/@antigen_epitope/abdb_v3/data_test/antigen_info_SI_interaction"

# List to hold all dataframes
dfs = []

# Iterate through each TSV file in the output directory and read them into a dataframe
for tsv_filename in os.listdir(output_dir):
    tsv_filepath = os.path.join(output_dir, tsv_filename)
    if tsv_filename.endswith('.tsv'):
        df = pd.read_csv(tsv_filepath, sep='\t')
        dfs.append(df)

# Concatenate all dataframes into a single dataframe
all_data_df = pd.concat(dfs, ignore_index=True)



# interacting info of antigen columns
ab_cols = ['G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N',
           'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']

# Add Status column
all_data_df['Status'] = np.where(
    all_data_df[ab_cols].sum(axis=1) == 0,
    'Non-Interacting',
    'Interacting'
)

# filter out non-Standard amino acid codes
standard_amino_acids = [
    'ALA', 'CYS', 'ASP', 'GLU', 'PHE',
    'GLY', 'HIS', 'ILE', 'LYS', 'LEU',
    'MET', 'ASN', 'PRO', 'GLN', 'ARG',
    'SER', 'THR', 'VAL', 'TRP', 'TYR'
]

# Filter the dataframe
all_data_df = all_data_df[all_data_df['AA'].isin(standard_amino_acids)]


# Define the directory where the merged file will be saved
save_dir = "/content/drive/My Drive/@antigen_epitope/abdb_v3/data_test/"

# Ensure the save directory exists
os.makedirs(save_dir, exist_ok=True)

# Save the concatenated dataframe to a CSV file
all_data_with_si_filepath = os.path.join(save_dir, 'all_data_with_SI.csv')
all_data_df.to_csv(all_data_with_si_filepath, index=False)
print(f"All data has been merged and saved to {all_data_with_si_filepath}")


In [ ]:
combined_df = all_data_df
unique_names_count = combined_df['name'].nunique()
unique_names_count

In [ ]:
number_of_rows = combined_df.shape[0]
print(number_of_rows)